# 🎯 Telco Churn Detection - Resource-Friendly GBT Model

**Objective**: Build an efficient churn prediction model optimized for limited resources

**Model**: Gradient Boosted Trees (GBT) with optimized hyperparameters

**Data Source**: Gold Layer (preprocessed and feature-engineered data)

---

In [1]:
# =========================================================
# TELCO CHURN – RESOURCE-FRIENDLY GBT MODEL
# Optimized for limited memory and CPU resources
# =========================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit
import time
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ------------------------
# CONFIGURATION MINIO
# ------------------------
MINIO_ACCESS_KEY = "minio"
MINIO_SECRET_KEY = "minio123"
MINIO_BUCKET = "telco-churn"
MINIO_ENDPOINT = "minio1:9000"

print("📋 Configuration MinIO:")
print(f"   → Bucket: {MINIO_BUCKET}")
print(f"   → Endpoint: {MINIO_ENDPOINT}")

📋 Configuration MinIO:
   → Bucket: telco-churn
   → Endpoint: minio1:9000


In [3]:
# ------------------------
# SPARK SESSION - RESOURCE-FRIENDLY CONFIGURATION
# ------------------------
print("\n🚀 Initializing Spark Session (Resource-Friendly Mode)...")

spark = (
    SparkSession.builder
    .appName("TelcoChurn_ResourceFriendly_GBT")
    .master("spark://spark-master:7077")

    # MinIO Configuration
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.endpoint", f"http://{MINIO_ENDPOINT}")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

    # Delta Lake
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

    # RESOURCE-FRIENDLY: Reduced memory and cores
    .config("spark.executor.instances", "1")  # Single executor
    .config("spark.executor.cores", "2")      # 2 cores per executor
    .config("spark.executor.memory", "2g")    # 2GB memory
    .config("spark.executor.memoryOverhead", "512m")

    # Driver Configuration
    .config("spark.driver.memory", "1g")      # 1GB driver memory
    .config("spark.driver.cores", "1")

    # Parallelism - Reduced for resource efficiency
    .config("spark.sql.shuffle.partitions", "4")  # Fewer partitions
    .config("spark.default.parallelism", "4")

    # Timeouts
    .config("spark.network.timeout", "600s")
    .config("spark.executor.heartbeatInterval", "30s")

    # Serialization
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    
    # Memory Management
    .config("spark.memory.fraction", "0.6")
    .config("spark.memory.storageFraction", "0.3")

    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"✅ Spark initialized successfully!")
print(f"   → App ID: {spark.sparkContext.applicationId}")
print(f"   → Executors: 1 (2 cores, 2GB RAM)")
print(f"   → Driver: 1 core, 1GB RAM")


🚀 Initializing Spark Session (Resource-Friendly Mode)...


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/20 15:23:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Spark initialized successfully!
   → App ID: app-20251220152315-0001
   → Executors: 1 (2 cores, 2GB RAM)
   → Driver: 1 core, 1GB RAM


In [4]:
# ------------------------
# LOAD GOLD LAYER DATA
# ------------------------
print("\n📂 Loading Gold Layer data...")
gold_path = f"s3a://{MINIO_BUCKET}/gold/telco_churn"

try:
    df = spark.read.format("delta").load(gold_path)
    total_rows = df.count()
    print(f"✅ Gold Layer loaded: {total_rows:,} rows")
    
    # Display schema
    print("\n📋 Dataset Schema:")
    df.printSchema()
    
    # Show sample
    print("\n📊 Sample Data:")
    df.select("customerID", "Churn", "features").show(5, truncate=True)
    
except Exception as e:
    print(f"❌ Error loading Gold Layer: {e}")
    raise


📂 Loading Gold Layer data...


25/12/20 15:23:26 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
25/12/20 15:23:34 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

✅ Gold Layer loaded: 10,777 rows

📋 Dataset Schema:
root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: integer (nullable = true)
 |-- Dependents: integer (nullable = true)
 |-- tenure: double (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: integer (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: integer (nullable = true)
 |-- OnlineBackup: integer (nullable = true)
 |-- DeviceProtection: integer (nullable = true)
 |-- TechSupport: integer (nullable = true)
 |-- StreamingTV: integer (nullable = true)
 |-- StreamingMovies: integer (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: integer (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: double (nullable = true)
 |-- Churn: integer (nullable = true)
 |-- HasInternetServ

+----------+-----+--------------------+
|customerID|Churn|            features|
+----------+-----+--------------------+
|9305-CDSKC|    1|(27,[0,3,7,10,15,...|
|7892-POOKP|    1|(27,[0,3,7,11,15,...|
|1066-JKSGK|    1|(27,[0,4,9,10,15,...|
|6467-CHFZW|    1|(27,[0,3,7,11,15,...|
|8773-HHUOZ|    1|(27,[0,4,8,13,15,...|
+----------+-----+--------------------+
only showing top 5 rows



In [5]:
# ------------------------
# PREPARE DATASET FOR MODELING
# ------------------------
print("\n🔧 Preparing dataset for modeling...")

# Select only necessary columns and cache for performance
df_model = df.select("features", "Churn").repartition(4).cache()
df_model.count()  # Materialize cache

print("✅ Dataset prepared and cached")

# Check class distribution
print("\n📊 Class Distribution:")
class_dist = df_model.groupBy("Churn").count().orderBy("Churn")
class_dist.show()

# Calculate class imbalance ratio
counts = class_dist.collect()
class_0 = counts[0]['count']
class_1 = counts[1]['count']
imbalance_ratio = class_0 / class_1 if class_1 > 0 else 1.0
print(f"   → Class 0 (No Churn): {class_0:,}")
print(f"   → Class 1 (Churn): {class_1:,}")
print(f"   → Imbalance Ratio: {imbalance_ratio:.2f}")


🔧 Preparing dataset for modeling...


✅ Dataset prepared and cached

📊 Class Distribution:
+-----+-----+
|Churn|count|
+-----+-----+
|    0| 5173|
|    1| 5604|
+-----+-----+

   → Class 0 (No Churn): 5,173
   → Class 1 (Churn): 5,604
   → Imbalance Ratio: 0.92


In [6]:
# ------------------------
# TRAIN/TEST SPLIT
# ------------------------
print("\n📊 Splitting data into Train (70%) and Test (30%)...")

train_df, test_df = df_model.randomSplit([0.7, 0.3], seed=42)

# Cache both datasets
train_df = train_df.cache()
test_df = test_df.cache()

train_count = train_df.count()
test_count = test_df.count()

print(f"✅ Data split completed:")
print(f"   → Training set: {train_count:,} rows ({train_count/total_rows*100:.1f}%)")
print(f"   → Test set: {test_count:,} rows ({test_count/total_rows*100:.1f}%)")

# Show training set class distribution
print("\n📊 Training Set Class Distribution:")
train_df.groupBy("Churn").count().orderBy("Churn").show()


📊 Splitting data into Train (70%) and Test (30%)...
✅ Data split completed:
   → Training set: 7,650 rows (71.0%)
   → Test set: 3,127 rows (29.0%)

📊 Training Set Class Distribution:
+-----+-----+
|Churn|count|
+-----+-----+
|    0| 3656|
|    1| 3994|
+-----+-----+



In [7]:
# =========================================================
#       🎯 RESOURCE-FRIENDLY GBT MODEL CONFIGURATION
# =========================================================
print("\n🎯 Configuring Resource-Friendly GBT Model...")

# Base GBT Classifier with conservative parameters
gbt = GBTClassifier(
    labelCol="Churn",
    featuresCol="features",
    seed=42
)

# RESOURCE-FRIENDLY PARAMETER GRID
# Smaller grid to reduce training time and memory usage
paramGrid = (
    ParamGridBuilder()
    .addGrid(gbt.maxDepth, [4, 5])           # Shallow trees (less memory)
    .addGrid(gbt.maxIter, [50, 80])          # Fewer iterations
    .addGrid(gbt.stepSize, [0.1, 0.15])      # Learning rate
    .addGrid(gbt.subsamplingRate, [0.8])     # Sample 80% of data
    .build()
)

print(f"✅ Parameter grid created: {len(paramGrid)} combinations")
print("\n📋 Hyperparameter Search Space:")
print("   → maxDepth: [4, 5]")
print("   → maxIter: [50, 80]")
print("   → stepSize: [0.1, 0.15]")
print("   → subsamplingRate: [0.8]")

# Evaluator - AUC-ROC for binary classification
evaluator = BinaryClassificationEvaluator(
    labelCol="Churn", 
    metricName="areaUnderROC"
)

# TrainValidationSplit (faster than CrossValidator)
# Uses 80% for training, 20% for validation
tvs = TrainValidationSplit(
    estimator=gbt,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    trainRatio=0.8,  # 80-20 split
    parallelism=2,   # Limited parallelism for resource efficiency
    seed=42
)

print("\n✅ TrainValidationSplit configured (80-20 split, parallelism=2)")


🎯 Configuring Resource-Friendly GBT Model...
✅ Parameter grid created: 8 combinations

📋 Hyperparameter Search Space:
   → maxDepth: [4, 5]
   → maxIter: [50, 80]
   → stepSize: [0.1, 0.15]
   → subsamplingRate: [0.8]

✅ TrainValidationSplit configured (80-20 split, parallelism=2)


In [8]:
# ------------------------
# TRAIN MODEL
# ------------------------
print("\n🚀 Starting model training...")
print("   ⏳ This may take a few minutes...")

t0 = time.time()

try:
    # Train the model with hyperparameter tuning
    tvs_model = tvs.fit(train_df)
    train_time = time.time() - t0
    
    print(f"\n✅ Model training completed!")
    print(f"   → Training time: {train_time/60:.2f} minutes ({train_time:.1f} seconds)")
    
    # Get best model
    best_model = tvs_model.bestModel
    
    # Display best hyperparameters
    print("\n🏆 Best Model Hyperparameters:")
    print(f"   → Max Depth: {best_model.getMaxDepth()}")
    print(f"   → Max Iterations: {best_model.getMaxIter()}")
    print(f"   → Step Size: {best_model.getStepSize()}")
    print(f"   → Subsampling Rate: {best_model.getSubsamplingRate()}")
    print(f"   → Number of Trees: {best_model.getNumTrees}")
    
except Exception as e:
    print(f"\n❌ Error during training: {e}")
    import traceback
    traceback.print_exc()
    raise


🚀 Starting model training...
   ⏳ This may take a few minutes...


25/12/20 15:24:53 WARN BlockManagerMaster: Failed to remove broadcast 719 with removeFromMaster = true - Block broadcast_719 does not exist
org.apache.spark.SparkException: Block broadcast_719 does not exist
	at org.apache.spark.errors.SparkCoreErrors$.blockDoesNotExistError(SparkCoreErrors.scala:318)
	at org.apache.spark.storage.BlockInfoManager.blockInfo(BlockInfoManager.scala:269)
	at org.apache.spark.storage.BlockInfoManager.removeBlock(BlockInfoManager.scala:547)
	at org.apache.spark.storage.BlockManager.removeBlockInternal(BlockManager.scala:2093)
	at org.apache.spark.storage.BlockManager.removeBlock(BlockManager.scala:2057)
	at org.apache.spark.storage.BlockManager.$anonfun$removeBroadcast$3(BlockManager.scala:2029)
	at org.apache.spark.storage.BlockManager.$anonfun$removeBroadcast$3$adapted(BlockManager.scala:2029)
	at scala.collection.Iterator.foreach(Iterator.scala:943)
	at scala.collection.Iterator.foreach$(Iterator.scala:943)
	at scala.collection.AbstractIterator.foreach(It


✅ Model training completed!
   → Training time: 2.50 minutes (150.1 seconds)

🏆 Best Model Hyperparameters:
   → Max Depth: 5
   → Max Iterations: 80
   → Step Size: 0.15
   → Subsampling Rate: 0.8
   → Number of Trees: 80


In [9]:
# ------------------------
# MODEL EVALUATION
# ------------------------
print("\n📊 Evaluating model on test set...")

# Make predictions
predictions = best_model.transform(test_df).cache()
predictions.count()  # Materialize cache

# Binary Classification Metrics
auc_evaluator = BinaryClassificationEvaluator(labelCol="Churn", metricName="areaUnderROC")
pr_evaluator = BinaryClassificationEvaluator(labelCol="Churn", metricName="areaUnderPR")

auc = auc_evaluator.evaluate(predictions)
pr_auc = pr_evaluator.evaluate(predictions)

# Multiclass Metrics
acc_evaluator = MulticlassClassificationEvaluator(labelCol="Churn", predictionCol="prediction", metricName="accuracy")
precision_evaluator = MulticlassClassificationEvaluator(labelCol="Churn", predictionCol="prediction", metricName="weightedPrecision")
recall_evaluator = MulticlassClassificationEvaluator(labelCol="Churn", predictionCol="prediction", metricName="weightedRecall")
f1_evaluator = MulticlassClassificationEvaluator(labelCol="Churn", predictionCol="prediction", metricName="f1")

accuracy = acc_evaluator.evaluate(predictions)
precision = precision_evaluator.evaluate(predictions)
recall = recall_evaluator.evaluate(predictions)
f1 = f1_evaluator.evaluate(predictions)

# Display Results
print("\n" + "="*70)
print("🎯 MODEL PERFORMANCE METRICS")
print("="*70)
print(f"\n📈 Binary Classification Metrics:")
print(f"   → AUC-ROC:  {auc:.4f}")
print(f"   → AUC-PR:   {pr_auc:.4f}")
print(f"\n📊 Multiclass Metrics:")
print(f"   → Accuracy:  {accuracy:.4f}")
print(f"   → Precision: {precision:.4f}")
print(f"   → Recall:    {recall:.4f}")
print(f"   → F1-Score:  {f1:.4f}")
print("\n" + "="*70)

# Performance assessment
if accuracy >= 0.80 and auc >= 0.85:
    print("\n🎉 EXCELLENT PERFORMANCE! Model meets production standards.")
elif accuracy >= 0.75 and auc >= 0.80:
    print("\n✅ GOOD PERFORMANCE! Model is suitable for deployment.")
else:
    print("\n⚠️ MODERATE PERFORMANCE. Consider further tuning.")


📊 Evaluating model on test set...

🎯 MODEL PERFORMANCE METRICS

📈 Binary Classification Metrics:
   → AUC-ROC:  0.8788
   → AUC-PR:   0.8583

📊 Multiclass Metrics:
   → Accuracy:  0.8059
   → Precision: 0.8091
   → Recall:    0.8059
   → F1-Score:  0.8049


🎉 EXCELLENT PERFORMANCE! Model meets production standards.


In [10]:
# ------------------------
# CONFUSION MATRIX
# ------------------------
print("\n📊 Confusion Matrix:")
conf_matrix = predictions.groupBy("Churn", "prediction").count().orderBy("Churn", "prediction")
conf_matrix.show()

# Calculate detailed metrics per class
conf_data = conf_matrix.collect()

# Parse confusion matrix
tn = fp = fn = tp = 0
for row in conf_data:
    if row['Churn'] == 0 and row['prediction'] == 0.0:
        tn = row['count']
    elif row['Churn'] == 0 and row['prediction'] == 1.0:
        fp = row['count']
    elif row['Churn'] == 1 and row['prediction'] == 0.0:
        fn = row['count']
    elif row['Churn'] == 1 and row['prediction'] == 1.0:
        tp = row['count']

print("\n📋 Confusion Matrix Breakdown:")
print(f"   → True Negatives (TN):  {tn:,}")
print(f"   → False Positives (FP): {fp:,}")
print(f"   → False Negatives (FN): {fn:,}")
print(f"   → True Positives (TP):  {tp:,}")

# Calculate per-class metrics
if (tp + fn) > 0:
    churn_recall = tp / (tp + fn)
    print(f"\n   → Churn Detection Rate (Recall for Class 1): {churn_recall:.4f}")

if (tp + fp) > 0:
    churn_precision = tp / (tp + fp)
    print(f"   → Churn Precision (Class 1): {churn_precision:.4f}")

if (tn + fp) > 0:
    specificity = tn / (tn + fp)
    print(f"   → Specificity (True Negative Rate): {specificity:.4f}")


📊 Confusion Matrix:
+-----+----------+-----+
|Churn|prediction|count|
+-----+----------+-----+
|    0|       0.0| 1126|
|    0|       1.0|  391|
|    1|       0.0|  216|
|    1|       1.0| 1394|
+-----+----------+-----+


📋 Confusion Matrix Breakdown:
   → True Negatives (TN):  1,126
   → False Positives (FP): 391
   → False Negatives (FN): 216
   → True Positives (TP):  1,394

   → Churn Detection Rate (Recall for Class 1): 0.8658
   → Churn Precision (Class 1): 0.7810
   → Specificity (True Negative Rate): 0.7423


In [11]:
# ------------------------
# FEATURE IMPORTANCE
# ------------------------
print("\n🔍 Feature Importance Analysis:")

# Get feature importances
feature_importances = best_model.featureImportances.toArray()
num_features = len(feature_importances)

# Sort features by importance
sorted_indices = feature_importances.argsort()[::-1]

# Display top 15 features
print("\n📊 Top 15 Most Important Features:")
print("   Rank | Feature Index | Importance")
print("   " + "-"*40)

for rank, idx in enumerate(sorted_indices[:15], 1):
    importance = feature_importances[idx]
    print(f"   {rank:2d}.  | Feature {idx:3d}   | {importance:.6f}")

# Calculate cumulative importance
cumulative_importance = 0
features_for_90_percent = 0

for idx in sorted_indices:
    cumulative_importance += feature_importances[idx]
    features_for_90_percent += 1
    if cumulative_importance >= 0.90:
        break

print(f"\n💡 Insight: Top {features_for_90_percent} features account for 90% of importance")


🔍 Feature Importance Analysis:

📊 Top 15 Most Important Features:
   Rank | Feature Index | Importance
   ----------------------------------------
    1.  | Feature  15   | 0.164910
    2.  | Feature  18   | 0.150814
    3.  | Feature  16   | 0.131659
    4.  | Feature  17   | 0.121417
    5.  | Feature   0   | 0.087728
    6.  | Feature   3   | 0.034534
    7.  | Feature  24   | 0.033010
    8.  | Feature   1   | 0.029046
    9.  | Feature  25   | 0.027968
   10.  | Feature  23   | 0.025026
   11.  | Feature   6   | 0.022314
   12.  | Feature   5   | 0.021064
   13.  | Feature  22   | 0.020831
   14.  | Feature   7   | 0.019839
   15.  | Feature  26   | 0.019640

💡 Insight: Top 15 features account for 90% of importance


In [12]:
# ------------------------
# SAVE MODEL
# ------------------------
print("\n💾 Saving model to MinIO...")

model_path = f"s3a://{MINIO_BUCKET}/models/gbt_churn_resource_friendly"

try:
    # Save the best model
    best_model.write().overwrite().save(model_path)
    
    print(f"✅ Model saved successfully!")
    print(f"   → Path: {model_path}")
    print(f"   → Format: Spark ML Model")
    
    # Also save validation metrics
    metrics_dict = {
        "auc_roc": float(auc),
        "auc_pr": float(pr_auc),
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
        "training_time_seconds": float(train_time),
        "test_samples": int(test_count),
        "train_samples": int(train_count)
    }
    
    # Convert to DataFrame and save
    metrics_df = spark.createDataFrame([metrics_dict])
    metrics_path = f"s3a://{MINIO_BUCKET}/models/gbt_churn_resource_friendly_metrics"
    metrics_df.write.mode("overwrite").json(metrics_path)
    
    print(f"   → Metrics saved: {metrics_path}")
    
except Exception as e:
    print(f"❌ Error saving model: {e}")
    import traceback
    traceback.print_exc()


💾 Saving model to MinIO...
✅ Model saved successfully!
   → Path: s3a://telco-churn/models/gbt_churn_resource_friendly
   → Format: Spark ML Model


25/12/20 15:28:57 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
                                                                                

   → Metrics saved: s3a://telco-churn/models/gbt_churn_resource_friendly_metrics


In [13]:
# ------------------------
# FINAL SUMMARY
# ------------------------
print("\n" + "="*70)
print("🎉 MODEL TRAINING COMPLETED SUCCESSFULLY!")
print("="*70)

print("\n📊 Summary:")
print(f"   → Model Type: Gradient Boosted Trees (GBT)")
print(f"   → Training Samples: {train_count:,}")
print(f"   → Test Samples: {test_count:,}")
print(f"   → Training Time: {train_time/60:.2f} minutes")
print(f"   → Best AUC-ROC: {auc:.4f}")
print(f"   → Best Accuracy: {accuracy:.4f}")
print(f"   → F1-Score: {f1:.4f}")

print("\n🎯 Resource Usage:")
print(f"   → Executors: 1 (2 cores, 2GB RAM)")
print(f"   → Driver: 1 core, 1GB RAM")
print(f"   → Total Memory: ~3.5GB")

print("\n💾 Model Artifacts:")
print(f"   → Model: {model_path}")
print(f"   → Metrics: {metrics_path}")

print("\n💡 Next Steps:")
print("   1. Deploy model to production environment")
print("   2. Set up real-time prediction API")
print("   3. Monitor model performance over time")
print("   4. Retrain periodically with new data")

print("\n" + "="*70)


🎉 MODEL TRAINING COMPLETED SUCCESSFULLY!

📊 Summary:
   → Model Type: Gradient Boosted Trees (GBT)
   → Training Samples: 7,650
   → Test Samples: 3,127
   → Training Time: 2.50 minutes
   → Best AUC-ROC: 0.8788
   → Best Accuracy: 0.8059
   → F1-Score: 0.8049

🎯 Resource Usage:
   → Executors: 1 (2 cores, 2GB RAM)
   → Driver: 1 core, 1GB RAM
   → Total Memory: ~3.5GB

💾 Model Artifacts:
   → Model: s3a://telco-churn/models/gbt_churn_resource_friendly
   → Metrics: s3a://telco-churn/models/gbt_churn_resource_friendly_metrics

💡 Next Steps:
   1. Deploy model to production environment
   2. Set up real-time prediction API
   3. Monitor model performance over time
   4. Retrain periodically with new data



In [14]:
# ------------------------
# CLEANUP
# ------------------------
print("\n🧹 Cleaning up resources...")

# Unpersist cached DataFrames
try:
    df_model.unpersist()
    train_df.unpersist()
    test_df.unpersist()
    predictions.unpersist()
    print("   → Cached DataFrames unpersisted")
except:
    pass

# Stop Spark session
print("\n🔴 Stopping Spark session...")
try:
    spark.stop()
    print("✅ Spark session stopped successfully")
    print("   → Resources released")
except Exception as e:
    print(f"⚠️ Error stopping Spark: {e}")

print("\n✅ Notebook execution completed!")


🧹 Cleaning up resources...
   → Cached DataFrames unpersisted

🔴 Stopping Spark session...
✅ Spark session stopped successfully
   → Resources released

✅ Notebook execution completed!
